In [ ]:
import requests
import xmltodict as xd
from dataclasses import dataclass, field
from datetime import datetime
from random import randint
from typing import List, Any, Optional, Dict
import logging
import sys
import base64
import sqlite3
from dataclasses import dataclass
from typing import List, Dict, Any
from datetime import datetime

# ----------------------------
# DB Manager Class
# ----------------------------
@dataclass
class DBManager:
    db_path: str = "hq_validation.db"

    def __post_init__(self):
        self.conn = sqlite3.connect(self.db_path)
        self.create_table()

    def create_table(self):
        query = """
        CREATE TABLE IF NOT EXISTS validation_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            request_id TEXT,
            bill_amt TEXT,
            customer_name TEXT,
            numran TEXT,
            status TEXT,
            errors TEXT,
            created_at TEXT
        )
        """
        self.conn.execute(query)
        self.conn.commit()

    def insert_log(
        self,
        request_data: Dict[str, Any],
        status: str,
        errors: List[str]
    ):
        query = """
        INSERT INTO validation_log 
        (request_id, bill_amt, customer_name, numran, status, errors, created_at)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """
        values = (
            request_data.get("POS_TAX_ID"),
            request_data.get("BILL_AMT"),
            request_data.get("CUSTOMER_NAME"),
            request_data.get("NUMRAN"),
            status,
            "; ".join(errors) if errors else None,
            datetime.now().isoformat(timespec="seconds")
        )
        self.conn.execute(query, values)
        self.conn.commit()

    def fetch_all(self):
        cur = self.conn.cursor()
        cur.execute("SELECT * FROM validation_log ORDER BY created_at DESC")
        return cur.fetchall()

# ----------------------------
# Setup Logging (file + console)
# ----------------------------
logger = logging.getLogger("HQAPI")
logger.setLevel(logging.DEBUG)

file_handler = logging.FileHandler("hq_request.log")
file_handler.setLevel(logging.INFO)

console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)

logger.addHandler(file_handler)
logger.addHandler(console_handler)


# ----------------------------
# Utility Functions
# ----------------------------
def generate_random_digits(length: int) -> str:
    return ''.join(str(randint(0, 9)) for _ in range(length))


def pad_left(data: str, length: int = 5) -> str:
    return data.zfill(length)


# ----------------------------
# Base Data Class
# ----------------------------
@dataclass
class BaseData:
    store_id: str = "09892"
    vendor_id: str = "82204"
    service_id: str = "00"
    item_name: str = "Test"
    data_fields: List[Any] = field(default_factory=lambda: [None]*8)
    amt_min: str = "1"
    amt_max: str = "90000"
    bill_amt: str = "50"
    cust_name: str = ""
    cust_addr_1: str = ""
    cust_addr_2: str = ""
    cust_addr_3: str = ""
    cust_phone_no: str = ""
    step: int = 1
    edit_out: Optional[str] = None
    zone: str = "1"
    employee_id: str = "0555505"
    pos_tax_id: str = "1537264827382"
    vat_amt: str = "0"
    rept_type: str = "H"
    payment_channel: str = "C05"
    date: str = field(default_factory=lambda: datetime.now().strftime("%Y/%m/%d"))
    time: str = field(default_factory=lambda: datetime.now().strftime("%X"))
    numran: str = field(default_factory=lambda: generate_random_digits(6))

    def merge_all_data(self) -> dict:
        data = {
            "STORE_ID": self.store_id,
            "ZONE": self.zone,
            "EMPLOYEE_ID": self.employee_id,
            "POS_TAX_ID": self.pos_tax_id,
            "VENDOR_ID": self.vendor_id,
            "SERVICE_ID": self.service_id,
            "ITEM_NAME": self.item_name,
            "VAT_AMT": self.vat_amt,
            "REPT_TYPE": self.rept_type,
            "PAYMENT_CHANNEL": self.payment_channel,
            "BILL_AMT": self.bill_amt,
            "AMT_MIN": self.amt_min,
            "AMT_MAX": self.amt_max,
            "CUSTOMER_NAME": self.cust_name,
            "CUSTOMER_ADDR_1": self.cust_addr_1,
            "CUSTOMER_ADDR_2": self.cust_addr_2,
            "CUSTOMER_ADDR_3": self.cust_addr_3,
            "CUSTOMER_TEL_NO": self.cust_phone_no,
            "BUS_DATE": self.date,
            "BUS_TIME": self.time,
            "NUMRAN": self.numran
        }
        for i, val in enumerate(self.data_fields):
            data[f"DATA_{i+1}"] = val
        return data


# ----------------------------
# XML Generator Class
# ----------------------------
@dataclass
class XMLGenerator:
    data: dict

    def generate_xml(self, action: str) -> str:
        xml_template = f"""<?xml version="1.0" encoding="UTF-8"?>
<HQ_REQUEST>
    <SERVICE_BOX>
        <ADDRESS>
            <VENDOR_ID>{self.data.get('VENDOR_ID')}</VENDOR_ID>
            <SERVICE_ID>{self.data.get('SERVICE_ID')}</SERVICE_ID>
            <METHOD>{action}</METHOD>
        </ADDRESS>
        <DATA>
"""
        for key, val in self.data.items():
            xml_template += f"            <{key}>{val}</{key}>\n"

        xml_template += "        </DATA>\n    </SERVICE_BOX>\n</HQ_REQUEST>"
        return xml_template

    def to_dict(self, xml_str: str) -> dict:
        return xd.parse(xml_str)


# ----------------------------
# API Client with Validation
# ----------------------------
class HQAPIError(Exception):
    """Custom exception สำหรับ API HQ"""
    pass


@dataclass
class HQAPIClient:
    endpoint: str
    timeout: int = 10  # seconds
    def Dataxml(self,Action):
        data = f"""<soapenv:Envelope xmlns:soapenv="http://schemas.xmlsoap.org/soap/envelope/" xmlns:por="http://portal.cs/">
        <soapenv:Header/>
        <soapenv:Body>
        <por:CSService>
        <!--Optional:-->
        <arg0><![CDATA[{Action}]]></arg0>
        </por:CSService>
        </soapenv:Body>
        </soapenv:Envelope>"""
        return data
    def send_request(self, xml_data: str) -> Optional[Dict[str, Any]]:
        logger.info(f"Sending request to {self.endpoint}")
        headers = {"Content-Type": "text/xml"}
        try:
            response = requests.request("POST",self.endpoint, headers=headers, data=self.Dataxml(xml_data).encode("utf-8"),timeout=self.timeout)
       #      response = requests.post(
       #          self.endpoint,
       #          data=xml_data.encode("utf-8"),
       #          headers=headers,
       #          timeout=self.timeout,
       #      )
            response.raise_for_status()

            logger.info(f"Response status: {response.status_code}")
            logger.debug(f"Raw response:\n{response.text}")

            parsed = xd.parse(response.text)["soap:Envelope"]["soap:Body"]["ns2:CSServiceResponse"]["return"]
            mystr_encoded = base64.b64decode(parsed).decode('utf-8') 
            parsed = xd.parse(mystr_encoded)
            return self.validate_response(parsed)

        except Exception as e:
            logger.exception(f"Request failed: {e}")
            return None

    def validate_response(self, response_dict: Dict[str, Any]) -> Dict[str, Any]:
        if not response_dict:
            raise HQAPIError("Empty response")

        if "HQ_RESPONSE" not in response_dict:
            raise HQAPIError("Invalid response format: missing 'HQ_RESPONSE' node")

        return response_dict["HQ_RESPONSE"]


# ----------------------------
# Response Validator
# ----------------------------
@dataclass
class ResponseValidator:
    request_data: dict
    response_data: dict
    errors: List[str] = field(default_factory=list)

    def validate_keys_exist(self, keys: List[str]) -> bool:
        for key in keys:
            if key not in self.response_data:
                self.errors.append(f"Missing key: {key}")
        return not self.errors

    def validate_values_match(self, keys: List[str]) -> bool:
        for key in keys:
            req_val = self.request_data.get(key)
            res_val = self.response_data.get(key)
            if str(req_val) != str(res_val):
                self.errors.append(f"Mismatch for {key}: req={req_val}, res={res_val}")
        return not self.errors

    def validate_status(self, status_key: str = "RESULT_CODE", success_value: str = "0") -> bool:
        result = self.response_data.get(status_key)
        if result != success_value:
            self.errors.append(f"Invalid status: {status_key}={result}")
            return False
        return True

    def validate(self) -> bool:
        required_keys = ["NUMRAN", "BILL_AMT", "CUSTOMER_NAME", "RESULT_CODE"]
        self.validate_keys_exist(required_keys)
        self.validate_values_match(["NUMRAN", "BILL_AMT", "CUSTOMER_NAME"])
        self.validate_status()

        if self.errors:
            logger.warning(f"Validation failed: {self.errors}")
            return False

        logger.info("Response validation: PASS")
        return True


In [ ]:
import sqlite3
from dataclasses import dataclass
from typing import List, Dict, Any
from datetime import datetime

# ----------------------------
# DB Manager Class
# ----------------------------
@dataclass
class DBManager:
    db_path: str = "hq_validation.db"

    def __post_init__(self):
        self.conn = sqlite3.connect(self.db_path)
        self.create_table()

    def create_table(self):
        query = """
        CREATE TABLE IF NOT EXISTS validation_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            request_id TEXT,
            bill_amt TEXT,
            customer_name TEXT,
            numran TEXT,
            status TEXT,
            errors TEXT,
            created_at TEXT
        )
        """
        self.conn.execute(query)
        self.conn.commit()

    def insert_log(
        self,
        request_data: Dict[str, Any],
        status: str,
        errors: List[str]
    ):
        query = """
        INSERT INTO validation_log 
        (request_id, bill_amt, customer_name, numran, status, errors, created_at)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """
        values = (
            request_data.get("POS_TAX_ID"),
            request_data.get("BILL_AMT"),
            request_data.get("CUSTOMER_NAME"),
            request_data.get("NUMRAN"),
            status,
            "; ".join(errors) if errors else None,
            datetime.now().isoformat(timespec="seconds")
        )
        self.conn.execute(query, values)
        self.conn.commit()

    def fetch_all(self):
        cur = self.conn.cursor()
        cur.execute("SELECT * FROM validation_log ORDER BY created_at DESC")
        return cur.fetchall()


In [ ]:
@dataclass
class ResponseValidator:
    request_data: dict
    response_data: dict
    errors: List[str] = field(default_factory=list)
    db: Optional[DBManager] = None   # <--- เพิ่ม DBManager

    def validate(self) -> bool:
        required_keys = ["NUMRAN", "BILL_AMT", "CUSTOMER_NAME", "RESULT_CODE"]
        self.validate_keys_exist(required_keys)
        self.validate_values_match(["NUMRAN", "BILL_AMT", "CUSTOMER_NAME"])
        self.validate_status()

        status = "PASS" if not self.errors else "FAIL"

        # ✅ เก็บลง Database
        if self.db:
            self.db.insert_log(
                request_data=self.request_data,
                status=status,
                errors=self.errors
            )

        if self.errors:
            logger.warning(f"Validation failed: {self.errors}")
            return False

        logger.info("Response validation: PASS")
        return True


In [ ]:
import xml.etree.ElementTree as ET
import xml.dom.minidom as minidom
import requests
import json
import logging
import pandas as pd


# -----------------------------
# Logger Setup
# -----------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
    handlers=[
        logging.FileHandler("app.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)


class DataManager:
    """Class สำหรับจัดการข้อมูล raw data"""

    def __init__(self, data: dict = None):
        self._data = data or {}
        self.logger = logging.getLogger(self.__class__.__name__)

    def set_data(self, key: str, value):
        """เพิ่ม/แก้ไขค่าใน data"""
        self._data[key] = value
        self.logger.info(f"Set data: {key} = {value}")

    def get_data(self, key: str, default=None):
        """ดึงค่าตาม key"""
        value = self._data.get(key, default)
        self.logger.debug(f"Get data: {key} -> {value}")
        return value

    def remove_data(self, key: str):
        """ลบข้อมูลตาม key"""
        if key in self._data:
            self.logger.info(f"Remove data: {key}")
            del self._data[key]

    def all_data(self) -> dict:
        """คืนค่าทุกข้อมูลใน dict"""
        return self._data


class XMLBuilder:
    """Class สำหรับสร้าง XML จาก data"""

    def __init__(self, root_name: str = "root"):
        self.root_name = root_name
        self.logger = logging.getLogger(self.__class__.__name__)

    def build_xml(self, data: dict, pretty: bool = True) -> str:
        """สร้าง XML string จาก dict"""
        root = ET.Element(self.root_name)
        self._dict_to_xml(root, data)
        xml_str = ET.tostring(root, encoding="utf-8")

        if pretty:
            xml_dom = minidom.parseString(xml_str)
            formatted = xml_dom.toprettyxml(indent="  ", encoding="utf-8").decode("utf-8")
            self.logger.info("Build formatted XML สำเร็จ")
            return formatted
        else:
            self.logger.info("Build raw XML สำเร็จ")
            return xml_str.decode("utf-8")

    def _dict_to_xml(self, parent, data: dict):
        """recursive dict → xml"""
        for key, value in data.items():
            elem = ET.SubElement(parent, key)
            if isinstance(value, dict):
                self._dict_to_xml(elem, value)
            else:
                elem.text = str(value)


class APIClient:
    """Class สำหรับส่ง Request API"""

    def __init__(self, base_url: str, data_manager: DataManager, root_name="Request", headers=None, timeout=10):
        self.base_url = base_url.rstrip("/")
        self.dm = data_manager
        self.xml_builder = XMLBuilder(root_name=root_name)
        self.headers = headers or {"Content-Type": "application/xml"}
        self.timeout = timeout
        self.logger = logging.getLogger(self.__class__.__name__)

    def send(self, endpoint: str, data: dict = None, pretty: bool = True):
        """
        ส่ง POST request พร้อม:
        1. ดึง data (ถ้าไม่ใส่ → ใช้ data ล่าสุด)
        2. แปลง data → XML (raw/pretty)
        3. แสดงตารางข้อมูลที่ส่ง
        4. ส่ง API และแสดง response เป็นตาราง
        5. เก็บ response ใน DataManager
        """
        # ดึง data
        payload = data or self.dm.all_data()
        if not payload:
            raise ValueError("ไม่มีข้อมูลที่จะส่ง (Data ว่างเปล่า)")

        # สร้าง XML
        xml_data = self.xml_builder.build_xml(payload, pretty=pretty)

        # แสดงตารางข้อมูลที่จะส่ง
        print("\n📤 ข้อมูลที่จะส่ง:")
        df_send = pd.DataFrame(payload.items(), columns=["Key", "Value"])
        print(df_send.to_string(index=False))

        # ส่ง request
        url = f"{self.base_url}/{endpoint.lstrip('/')}"
        self.logger.info(f"POST -> {url}")

        try:
            response = requests.post(url, data=xml_data, headers=self.headers, timeout=self.timeout)
            response.raise_for_status()
        except Exception as e:
            self.logger.error(f"API Error: {e}")
            return {"error": str(e)}

        # แปลง response
        result = {
            "status": response.status_code,
            "headers": dict(response.headers),
            "text": response.text
        }

        # แสดง response เป็นตาราง
        print("\n📥 ข้อมูลที่ได้รับกลับมา:")
        df_resp = pd.DataFrame(result.items(), columns=["Key", "Value"])
        print(df_resp.to_string(index=False))

        # เก็บ response ลง DataManager
        self.dm.set_data("last_response", result)

        return result


# -----------------------------
# ตัวอย่างการใช้งาน
# -----------------------------
if __name__ == "__main__":
    # 1. สร้าง DataManager
    dm = DataManager()
    dm.set_data("member_id", "12345")
    dm.set_data("name", "ซีซาร์")
    dm.set_data("point", 200)

    # 2. ใช้ APIClient ส่ง API
    client = APIClient(base_url="https://httpbin.org", data_manager=dm, root_name="MemberInfo")
    result = client.send("/post", pretty=True)

    print("\n✅ Response เก็บใน DataManager แล้ว:")
    print(dm.get_data("last_response"))


2025-09-18 03:12:45,884 [INFO] DataManager - Set data: member_id = 12345
2025-09-18 03:12:45,893 [INFO] DataManager - Set data: name = ซีซาร์
2025-09-18 03:12:45,894 [INFO] DataManager - Set data: point = 200
2025-09-18 03:12:45,909 [INFO] XMLBuilder - Build formatted XML สำเร็จ
2025-09-18 03:12:46,014 [INFO] APIClient - POST -> https://httpbin.org/post



📤 ข้อมูลที่จะส่ง:
      Key  Value
member_id  12345
     name ซีซาร์
    point    200


2025-09-18 03:12:47,563 [ERROR] APIClient - API Error: 503 Server Error: Service Temporarily Unavailable for url: https://httpbin.org/post



✅ Response เก็บใน DataManager แล้ว:
None


In [ ]:
from decimal import Decimal, ROUND_HALF_UP
from typing import Union

def normalize_decimal(value: Union[int, float, Decimal]) -> Decimal:
    """
    Normalize a numeric value so that its decimal part is restricted to
    one of: .00, .25, .50, .75.

    Args:
        value (int | float | Decimal): The numeric value to normalize.

    Returns:
        Decimal: The normalized number with two decimal places.

    Raises:
        TypeError: If the input is not int, float, or Decimal.
    """
    if not isinstance(value, (int, float, Decimal)):
        raise TypeError("Value must be int, float, or Decimal.")

    # Convert to Decimal with 2 decimal precision
    dec_value = Decimal(str(value)).quantize(Decimal("0.00"), rounding=ROUND_HALF_UP)

    integer_part = int(dec_value)
    decimal_part = dec_value - Decimal(integer_part)

    # Possible decimal steps
    steps = [Decimal("0.00"), Decimal("0.25"), Decimal("0.50"), Decimal("0.75")]

    # Find nearest step
    nearest = min(steps, key=lambda x: abs(decimal_part - x))

    # Special case: if rounding pushes above 0.875 → next integer
    if decimal_part >= Decimal("0.875"):
        integer_part += 1
        nearest = Decimal("0.00")

    return Decimal(integer_part) + nearest


In [ ]:
dm.all_data()


{'member_id': '12345', 'name': 'ซีซาร์', 'point': 200}

In [ ]:
import random
from faker import Faker
from datetime import datetime
fake = Faker("th_TH") 

STORE_ID = "09884"
MACHINE_ID =''.join(random.choices("1234", k=1))
VENDOR_CODE =0
SERVICE_ID = 0
VENDOR_ID =0

ITEM_NAME=0
COMMON_TX_ID=str(randint(0,100))
SHIFT_ID =''.join(random.choices("123456789", k=1))
TX_TYPE ="N"
ITEM_SELECTIC="N"
PAYMENT_TYPE ="001"
ZONE = "1"
EMPLOYEE_ID = ''.join(random.choices("056789", k=7))
POS_TAX_ID = ''.join(random.choices("0123456789", k=13))
RECEIPT_NO = str(randint(0,1000))
RECEIPT_TYPE = "H"
PAYMENT_CHANNEL = 'C05'

SERVICE_ID_OUT = 0
VENDOR_ID_OUT =0
TX_ID=0
CLIENT_SERVICE_SEQUENCE=0
CLIENT_SEQUENCE_NO=0
DATA_1_TYPE = 0
DATA_2_TYPE = 0
DATA_3_TYPE = 0
DATA_4_TYPE = 0
DATA_5_TYPE = 0
DATA_6_TYPE = 0
DATA_7_TYPE = 0
DATA_9_TYPE = 0
DATA_1 = 0
DATA_2 = 0
DATA_3 = 0
DATA_4 = 0
DATA_5 = 0
DATA_6 = 0
DATA_7 = 0
DATA_9 = 0

BILL_AMT_MAX =0
BILL_AMT_MIN=0
BILL_AMT_EDIT=0
BILL_AMT =0
BILL_AMT_ROUND =0
BILL_AMT_VAT ="0"

CUST_NAME = fake.first_name()
CUST_ADDR_1 = fake.address()
CUST_ADDR_2 = fake.address()
CUST_ADDR_3 = 0
CUST_PHONE_NO = fake.phone_number()
CUST_TAG_NAME = 0

STEP=0
HOURS=0
MINUTES=0
HOURS_NEW=0
MINUTES_NEW=0


DATE_SYS = (datetime.now()).strftime("%Y/%m/%d")
TIME_SYS = (datetime.now()).strftime("%X")
DATE_BUS = (datetime.now()).strftime("%Y/%m/%d")
TIME_BUS = (datetime.now()).strftime("%X")


In [ ]:
import random
from random import randint
from faker import Faker
from datetime import datetime
import copy

class DataStore:
    def __init__(self):
        self.fake = Faker(THAI)
        self._saved_data = {}   # เก็บข้อมูลที่ save แล้ว
        self._save_index = ZERO    # running index

        # กำหนดค่าเริ่มต้น
        self._current_data = {
            "STORE_ID": "09884",
            "MACHINE_ID": ''.join(random.choices("1234", k=1)),
            "VENDOR_CODE": ZERO,
            "SERVICE_ID": ZERO,
            "VENDOR_ID": ZERO,

            "ITEM_NAME": ZERO,
            "COMMON_TX_ID": str(randint(ZERO, 100)),
            "SHIFT_ID": ''.join(random.choices("123456789", k=1)),
            "TX_TYPE": N,
            "ITEM_SELECTIC": N,
            "PAYMENT_TYPE": CASH,
            "ZONE": 1,
            "EMPLOYEE_ID": ''.join(random.choices("056789", k=7)),
            "POS_TAX_ID": ''.join(random.choices("0123456789", k=13)),
            "RECEIPT_NO": str(randint(ZERO, 1000)),
            "RECEIPT_TYPE": H,
            "PAYMENT_CHANNEL": POS,
	     "PRINTSLIP":"",
            "SUCCESS":"ZERO",
	     "CODE":"",
	     "DESCRIPTOR":"",

            "SERV_ID": ZERO,
            "VENDOR_ID_OUT": ZERO,
            "TX_ID": ZERO,
            "CLIENT_SERVICE_SEQUENCE": ZERO,
            "CLIENT_SEQUENCE_NO": ZERO,
            "DATA_1_NO": ZERO,
            "DATA_2_NO": ZERO,
            "DATA_3_NO": ZERO,
            "DATA_4_NO": ZERO,
            "DATA_5_NO": ZERO,
            "DATA_6_NO": ZERO,
            "DATA_7_NO": ZERO,
            "DATA_9_NO": ZERO,
            "DATA_1": ZERO,
            "DATA_2": ZERO,
            "DATA_3": ZERO,
            "DATA_4": ZERO,
            "DATA_5": ZERO,
            "DATA_6": ZERO,
            "DATA_7": ZERO,
            "DATA_9": ZERO,
            
	     "BILL_AMT_ABOVE_MAX": 90001,
            "BILL_AMT_MAX": 90000,
            "BILL_AMT_CENTER": 49000,
            "BILL_AMT_MIN": 1,
	     "BILL_AMT_BELOW_MIN": 0.75,
	     "BILL_AMT_NULL": "",
            "BILL_AMT_ZERO": ZERO,
            "BILL_AMT_DECIMAL": 100.25,
            "BILL_AMT_NON_DECIMAL": 100.26,
            "BILL_AMT_EDIT": 100,
            "BILL_AMT": 100,
            "BILL_AMT_ROUND": 100,
            "BILL_AMT_VAT": ZERO,
	     "VAT":"",
	     "FEE":"",
	     "FEE_VAT":"",
         
            "CUSTOMER_NAME": self.fake.first_name(),
            "CUSTOMER_ADDR_1": self.fake.address(),
            "CUSTOMER_ADDR_2": self.fake.address(),
            "CUSTOMER_ADDR_3": "",
            "CUSTOMER_TEL_NO": self.fake.phone_number(),
            "CUSTOMER_TAX_ID": "",
	     "CUSTOMER_BRANCH_CODE":"",
	     "CUSTOMER_RECEIPT_NAME":"",
	     "CUSTOMER_RECEIPT_ADDR":"",
            "ACTION": "INIT",
            "FUNTION": "INIT",
            "STEP": ZERO,
            "HOURS": ZERO,
            "MINUTES": ZERO,
            "HOURS_NEW": ZERO,
            "MINUTES_NEW": ZERO,
            "ACCT_NO":"",
            "DATE_SYS": datetime.now().strftime("%Y/%m/%d"),
            "TIME_SYS": datetime.now().strftime("%X"),
            "DATE_BUS": datetime.now().strftime("%Y/%m/%d"),
            "TIME_BUS": datetime.now().strftime("%X"),
            


	 }


    # ------------------------------
    # Function จัดการข้อมูล
    # ------------------------------
    def update(self, key, value):
        """แก้ไขค่าปัจจุบัน"""
        if key in self._current_data:
            self._current_data[key] = value
        else:
            self._current_data[key] = value
            print(f"{key} ทำการสร้างตัวเเปรใหม่")
    def Generator_DATA(self,value):
        """แก้ไขค่าปัจจุบัน"""
        if isinstance(value, int):
            return ''.join(random.choices("0123456789", k=value))
        elif isinstance(value, str):
            return value
        elif isinstance(value, list):
            return value[0] if value else None
        else:
            return None
    def add_field(self, key, value):
        """เพิ่มตัวแปรใหม่"""
        self._current_data[key] = value
    def refresh_date_time(self):
        """แก้ไขค่าปัจจุบัน"""
        self._current_data["DATE_SYS"] = datetime.now().strftime("%Y/%m/%d")
        self._current_data["TIME_SYS"] = datetime.now().strftime("%X")
        self._current_data["DATE_BUS"] = datetime.now().strftime("%Y/%m/%d")
        self._current_data["TIME_BUS"] = datetime.now().strftime("%X")

    def refresh_bill_amt(self):
        """แก้ไขค่าปัจจุบัน"""
        self._current_data["BILL_AMT_ABOVE_MAX"] = self._current_data["BILL_AMT_MAX"]+1
        self._current_data["BILL_AMT_BELOW_MIN"] = self._current_data["BILL_AMT_MIN"]-0.5
        self._current_data["BILL_AMT_EDIT"] = self._current_data["BILL_AMT"]+random.choice([x for x in range(1, 99) if x % 10 == 0])
        self._current_data["BILL_AMT_ROUND"] = normalize_decimal(self._current_data["BILL_AMT"])
        self._current_data["BILL_AMT_DECIMAL"] = self._current_data["BILL_AMT"]+(random.choices([0.25, 0.50, 0.75], k=1)/100)
        self._current_data["BILL_AMT_NON_DECIMAL"] = self._current_data["BILL_AMT"]+(random.choice([x for x in range(1, 99) if x % 25 != 0])/100)
    def refresh_value(self):
        """แก้ไขค่าปัจจุบัน"""
        key = ["DATA_1","DATA_2","DATA_3","DATA_4","DATA_5","DATA_6"]
        key_value = ["DATA_1_NO","DATA_2_NO","DATA_3_NO","DATA_4_NO","DATA_5_NO","DATA_6_NO"]
        for k,kv in zip(key,key_value):
           if k in self._current_data:
               self._current_data[kv] = self.Generator_DATA(self._current_data[k])
           else:
               raise KeyError(f"{key} ไม่พบใน current data")
    def get_current(self):
        """คืนค่าปัจจุบัน"""
        return self._current_data

    def save(self):
        """บันทึก snapshot (แก้ไขไม่ได้)"""
        self._save_index += 1
        self._saved_data[self._save_index] = copy.deepcopy(self._current_data)
        return self._save_index

    def get_all_value(self, index=None):
        """ดึงค่าที่ save แล้ว"""
        if index:
            return self._saved_data.get(index, None)
        return self._saved_data


# ------------------------------
# ตัวอย่างการใช้งาน
# ------------------------------
# if __name__ == "__main__":
#     ds = DataStore()

#     print("🔹 Current Data:")
#     print(ds.get_current())

#     # แก้ค่า
#     ds.update("ITEM_NAME", "สินค้าA")
#     ds.add_field("NEW_FIELD", "ค่าใหม่")

#     # save
#     idx = ds.save()
#     print(f"\n✅ บันทึกข้อมูล index {idx}")
#     print(ds.get_all_value(idx))

#     # แก้ต่อ
#     ds.update("ITEM_NAME", "สินค้าB")
#     idx2 = ds.save()
#     print(f"\n✅ บันทึกข้อมูล index {idx2}")
#     print(ds.get_all_value(idx2))

#     print("\n📦 All Saved Data:")
#     print(ds.get_all_value())


In [ ]:
import xml.etree.ElementTree as ET
import xml.dom.minidom as minidom
import logging
from typing import Any, Dict, Optional, Union

logger = logging.getLogger("XMLFormatter")
logger.setLevel(logging.INFO)
if not logger.handlers:
    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(name)s - %(message)s"))
    logger.addHandler(ch)


TemplateSpec = Dict[str, Any]  # nested dict spec for xml template


class XMLFormatter:
    """
    XMLFormatter สร้าง XML จาก nested template spec และแม็ปค่าใน data dict

    Template spec format:
    - Nested dict: each key => element name
    - Leaf value: string key name ที่จะดึงจาก data dict OR a literal by using {"_literal": "text"}
    - หากต้องการ element ซ้ำ (list) ให้ใช้ list of specs, เช่น:
        "Items": [ {"Item": {"Code":"ITEM_CODE"}}, {"Item": {"Code":"ANOTHER"}} ]
    Example template:
    {
        "Request": {
            "Header": {
                "StoreId": "STORE_ID",
                "MachineId": "MACHINE_ID"
            },
            "Body": {
                "Transaction": {
                    "TX_ID": "TX_ID",
                    "Amount": "BILL_AMT"
                }
            }
        }
    }
    """

    def __init__(self) -> None:
        self._templates: Dict[int, TemplateSpec] = {}
        self._init_default_templates()

    # -------------------------
    # Public API
    # -------------------------
    def build(self, format_id: int, data: Dict[str, Any], pretty: bool = True) -> str:
        """
        สร้าง XML string จาก template ที่ระบุและ data
        :param format_id: id ของ template (1..9)
        :param data: dict mapping field names -> values
        :param pretty: ถ้า True จะคืนค่า pretty-printed XML
        :return: XML string
        """
        if format_id not in self._templates:
            raise ValueError(f"Template {format_id} not found")

        spec = self._templates[format_id]
        root_tag = next(iter(spec))  # assume top-level single root
        root_spec = spec[root_tag]

        root_elem = ET.Element(root_tag)
        self._spec_to_xml(root_elem, root_spec, data)

        raw = ET.tostring(root_elem, encoding="utf-8")
        if pretty:
            dom = minidom.parseString(raw)
            pretty_xml = dom.toprettyxml(indent="  ", encoding="utf-8").decode("utf-8")
            logger.info(f"Built pretty XML for format {format_id}")
            return pretty_xml
        else:
            logger.info(f"Built raw XML for format {format_id}")
            return raw.decode("utf-8")

    def add_template(self, format_id: int, spec: TemplateSpec) -> None:
        """เพิ่ม template ใหม่ (ทับได้ถ้าต้องการ)"""
        self._templates[format_id] = spec
        logger.info(f"Template {format_id} added/updated")

    def get_template(self, format_id: int) -> Optional[TemplateSpec]:
        return self._templates.get(format_id)

    def remove_template(self, format_id: int) -> None:
        if format_id in self._templates:
            del self._templates[format_id]
            logger.info(f"Template {format_id} removed")

    # -------------------------
    # Internal helpers
    # -------------------------
    def _spec_to_xml(self, parent: ET.Element, spec: Any, data: Dict[str, Any]) -> None:
        """
        Recursive: แปลง spec -> XML under parent
        spec สามารถเป็น:
          - dict => สร้าง child elements ตาม keys
          - list => loop สร้าง elements ตาม items ของ list
          - str => ถ้ามี key ใน data ให้ใช้ data[key] เป็น text, ถ้าไม่มีกลับเป็น empty string
          - {"_literal": "some text"} => ใช้ literal text
        """
        if isinstance(spec, dict):
            for tag, subspec in spec.items():
                # If the tag is special (like attributes) we could extend - for now treat as normal tag
                if isinstance(subspec, list):
                    # list => สร้าง repeated elements under tag
                    parent_section = ET.SubElement(parent, tag)
                    for item in subspec:
                        # each item expected as a dict with single element name (or nested)
                        if isinstance(item, dict):
                            # if item is {"Item": {...}} -> produce Item children
                            for inner_tag, inner_spec in item.items():
                                node = ET.SubElement(parent_section, inner_tag)
                                self._spec_to_xml(node, inner_spec, data)
                        else:
                            # primitive list item
                            node = ET.SubElement(parent_section, "Value")
                            node.text = str(item)
                else:
                    node = ET.SubElement(parent, tag)
                    self._spec_to_xml(node, subspec, data)
        elif isinstance(spec, list):
            # create multiple children with generic name "Item" unless handled above
            for item in spec:
                node = ET.SubElement(parent, "Item")
                self._spec_to_xml(node, item, data)
        elif isinstance(spec, str):
            # spec is a key name to lookup in data
            val = data.get(spec, "")
            node_text = "" if val is None else str(val)
            parent.text = node_text
        elif isinstance(spec, (int, float)):
            parent.text = str(spec)
        elif isinstance(spec, dict) and "_literal" in spec:
            parent.text = str(spec["_literal"])
        else:
            # unsupported leaf type -> set empty
            parent.text = ""

    # -------------------------
    # Default 9 templates (ตัวอย่าง)
    # -------------------------
    def _init_default_templates(self) -> None:
        """
        สร้างตัวอย่าง template 1..9 — ซีซาร์สามารถแก้หรือเพิ่มเองได้
        แต่ละ template ใช้ field ที่พบบ่อยจาก DataStore
        """
        # Template 1: Minimal header+body
        t1 = {
            "Request": {
                "Header": {
                    "StoreId": "STORE_ID",
                    "MachineId": "MACHINE_ID",
                    "Date": "DATE_SYS",
                    "Time": "TIME_SYS"
                },
                "Body": {
                    "Transaction": {
                        "TX_ID": "TX_ID",
                        "ReceiptNo": "RECEIPT_NO",
                        "Amount": "BILL_AMT"
                    }
                }
            }
        }

        # Template 2: includes customer info
        t2 = {
            "PaymentRequest": {
                "Meta": {
                    "Vendor": "VENDOR_ID",
                    "Service": "SERVICE_ID",
                    "Zone": "ZONE"
                },
                "Customer": {
                    "Name": "CUST_NAME",
                    "Phone": "CUST_PHONE_NO",
                    "Addr1": "CUST_ADDR_1"
                },
                "Payload": {
                    "Amount": "BILL_AMT",
                    "PaymentType": "PAYMENT_TYPE"
                }
            }
        }

        # Template 3: Repeated item example
        t3 = {
            "Order": {
                "Header": {
                    "Store": "STORE_ID",
                    "Shift": "SHIFT_ID"
                },
                "Items": [
                    {"Item": {"Name": "ITEM_NAME", "Select": "ITEM_SELECTIC"}}
                ],
                "Total": {"Amount": "BILL_AMT"}
            }
        }

        # Template 4: Full transaction with sequences
        t4 = {
            "TxnRequest": {
                "ClientSeq": "CLIENT_SEQUENCE_NO",
                "ClientServiceSeq": "CLIENT_SERVICE_SEQUENCE",
                "ServiceOut": "SERVICE_ID_OUT",
                "VendorOut": "VENDOR_ID_OUT",
                "Details": {
                    "Data1": "DATA_1",
                    "Data2": "DATA_2",
                    "Data3": "DATA_3"
                }
            }
        }

        # Template 5: Receipt style
        t5 = {
            "Receipt": {
                "Header": {
                    "TaxId": "POS_TAX_ID",
                    "ReceiptNo": "RECEIPT_NO",
                    "ReceiptType": "RECEIPT_TYPE"
                },
                "Customer": {
                    "Name": "CUST_NAME",
                    "Phone": "CUST_PHONE_NO"
                },
                "Amounts": {
                    "Bill": "BILL_AMT",
                    "BillRound": "BILL_AMT_ROUND",
                    "VAT": "BILL_AMT_VAT"
                }
            }
        }

        # Template 6: Simple ping / heartbeat
        t6 = {
            "Ping": {
                "Store": "STORE_ID",
                "Time": "TIME_SYS",
                "Date": "DATE_SYS"
            }
        }

        # Template 7: Complex nested
        t7 = {
            "ComplexRequest": {
                "Header": {
                    "Store": "STORE_ID",
                    "Machine": "MACHINE_ID",
                    "Employee": "EMPLOYEE_ID"
                },
                "Body": {
                    "Step": "STEP",
                    "Timing": {
                        "Hours": "HOURS",
                        "Minutes": "MINUTES",
                        "HoursNew": "HOURS_NEW",
                        "MinutesNew": "MINUTES_NEW"
                    },
                    "CustomTag": {"_literal": "This is a literal value"}
                }
            }
        }

        # Template 8: Minimal IDs
        t8 = {
            "IDs": {
                "CommonTx": "COMMON_TX_ID",
                "TxType": "TX_TYPE",
                "PaymentChannel": "PAYMENT_CHANNEL"
            }
        }

        # Template 9: All-in-one (flatten many fields)
        t9 = {
            "Envelope": {
                "StoreId": "STORE_ID",
                "MachineId": "MACHINE_ID",
                "VendorCode": "VENDOR_CODE",
                "ServiceId": "SERVICE_ID",
                "VendorId": "VENDOR_ID",
                "PosTax": "POS_TAX_ID",
                "Cust": {
                    "Name": "CUST_NAME",
                    "Addr1": "CUST_ADDR_1",
                    "Phone": "CUST_PHONE_NO"
                },
                "Amounts": {
                    "Bill": "BILL_AMT",
                    "Max": "BILL_AMT_MAX",
                    "Min": "BILL_AMT_MIN"
                },
                "Meta": {
                    "DateBus": "DATE_BUS",
                    "TimeBus": "TIME_BUS"
                }
            }
        }

        # register templates 1..9
        for i, t in enumerate([t1, t2, t3, t4, t5, t6, t7, t8, t9], start=1):
            self._templates[i] = t
        logger.info("Initialized default 9 XML templates")


# -------------------------
# Example usage
# -------------------------
if __name__ == "__main__":
    # สมมติ data มาจาก DataStore.get_current()
    sample_data = {
        "STORE_ID": "09884",
        "MACHINE_ID": "3",
        "DATE_SYS": "2025/09/14",
        "TIME_SYS": "12:34:56",
        "TX_ID": 123,
        "RECEIPT_NO": "789",
        "BILL_AMT": 250.75,
        "CUST_NAME": "สมชาย",
        "CUST_PHONE_NO": "081-234-5678",
        "ITEM_NAME": "สินค้าA",
        "ITEM_SELECTIC": "Y",
        "EMPLOYEE_ID": "0567890",
        "HOURS": 10,
        "MINUTES": 30
    }

    xf = XMLFormatter()
    xml1 = xf.build(1, sample_data, pretty=True)
    print(xml1)

    xml3 = xf.build(3, sample_data, pretty=True)
    print(xml3)


2025-09-15 23:03:20,267 [INFO] XMLFormatter - Initialized default 9 XML templates
2025-09-15 23:03:20,274 [INFO] XMLFormatter - Built pretty XML for format 1
2025-09-15 23:03:20,276 [INFO] XMLFormatter - Built pretty XML for format 3


<?xml version="1.0" encoding="utf-8"?>
<Request>
  <Header>
    <StoreId>09884</StoreId>
    <MachineId>3</MachineId>
    <Date>2025/09/14</Date>
    <Time>12:34:56</Time>
  </Header>
  <Body>
    <Transaction>
      <TX_ID>123</TX_ID>
      <ReceiptNo>789</ReceiptNo>
      <Amount>250.75</Amount>
    </Transaction>
  </Body>
</Request>

<?xml version="1.0" encoding="utf-8"?>
<Order>
  <Header>
    <Store>09884</Store>
    <Shift/>
  </Header>
  <Items>
    <Item>
      <Name>สินค้าA</Name>
      <Select>Y</Select>
    </Item>
  </Items>
  <Total>
    <Amount>250.75</Amount>
  </Total>
</Order>



In [ ]:
import xmltodict
import requests
import base64
from typing import Optional
from decimal import Decimal, ROUND_HALF_UP
from typing import Union
from faker import Faker
import random
from random import randint
from datetime import datetime
import copy
from pprint import pprint
import pandas as pd
from itables import show
import xmltodict
from collections import defaultdict
from enum import Enum
from icecream import ic

DATAEXCHANGE="DATAEXCHANGE"
CANCEL="CANCEL"
DATAEXCHANGECONFIRM="DATAEXCHANGECONFIRM"
REPRINTSLIP="REPRINTSLIP"
OR= "OR"
ORCANCEL="ORCANCEL"
ORCONFIRM="ORCONFIRM"
INQUIRY="INQUIRY"

ACTION = "ACTION"
FUNTION= "FUNTION"
REQUEST = "REQUEST"
RESPONSE = "RESPONSE"

N="N"
R ="R"
FULLFROM = "F"

H="H"
# TRUE =  1= True 
# FALSE  = 0= False
POS ='C05'

CASH = "001"

ZERO=0
THAI ="th_TH"

STORE_ID = "STORE_ID"
MACHINE_ID ="MACHINE_ID"
VENDOR_CODE ="VENDOR_CODE"
SERVICE_ID = "SERVICE_ID"
VENDOR_ID ="VENDOR_ID"

ITEM_NAME="ITEM_NAME"
COMMON_TX_ID="COMMON_TX_ID"
SHIFT_ID ="SHIFT_ID"
TX_TYPE ="TX_TYPE"
ITEM_SELECTIC="ITEM_SELECTIC"
PAYMENT_TYPE ='PAYMENT_TYPE'
ZONE = "ZONE"
EMPLOYEE_ID = "EMPLOYEE_ID"
POS_TAX_ID = "POS_TAX_ID"
RECEIPT_NO = "RECEIPT_NO"
RECEIPT_TYPE = "RECEIPT_TYPE"
PAYMENT_CHANNEL = "PAYMENT_CHANNEL"

SERVICE_ID_OUT = "SERVICE_ID_OUT"
VENDOR_ID_OUT ="VENDOR_ID_OUT"
TX_ID="TX_ID"
CLIENT_SERVICE_SEQUENCE="CLIENT_SERVICE_SEQUENCE"
CLIENT_SEQUENCE_NO="CLIENT_SEQUENCE_NO"

DATA_1 = "DATA_1_NO"
DATA_2 = "DATA_2_NO"
DATA_3 = "DATA_3_NO"
DATA_4 = "DATA_4_NO"
DATA_5 = "DATA_5_NO"
DATA_6 = "DATA_6_NO"
DATA_7 = "DATA_7_NO"
DATA_9 = "DATA_9_NO"

BILL_AMT_MAX ="BILL_AMT_MIN"
BILL_AMT_MIN="BILL_AMT_MIN"
BILL_AMT_EDIT="BILL_AMT_EDIT"
BILL_AMT ="BILL_AMT"
BILL_AMT_ROUND ="BILL_AMT_ROUND"
BILL_AMT_VAT ="BILL_AMT_VAT"
CLIENT_SEQUENCE_NO_OR ="CLIENT_SEQUENCE_NO_OR"
CUST_NAME = "CUST_NAME"
CUST_ADDR_1 = "CUST_ADDR_1"
CUST_ADDR_2 = "CUST_ADDR_2"
CUST_ADDR_3 = "CUST_ADDR_3"
CUST_PHONE_NO = "CUST_PHONE_NO"
CUST_TAG_NAME = "CUST_TAG_NAME"
UNKNOWN = ""
STEP="STEP"
HOURS="HOURS"
MINUTES="MINUTES"
HOURS_NEW="HOURS_NEW"
MINUTES_NEW="MINUTES_NEW"
CLIENT_SERVICE_SEQUENCE_OR="CLIENT_SERVICE_SEQUENCE_OR"
CLIENT_SEQUENCE="CLIENT_SEQUENCE"
CLIENT_NO ="CLIENT_NO"
ERROR ="ERROR"
DATE_SYS = "DATE_SYS"
TIME_SYS = "TIME_SYS"
DATE_BUS = "DATE_BUS"
TIME_BUS = "TIME_BUS"
class TemplateType(str, Enum):
    DATAEXCHANGE = "DATAEXCHANGE"
    CANCEL = "CANCEL"
    DATAEXCHANGECONFIRM = "DATAEXCHANGECONFIRM"
    REPRINTSLIP = "REPRINTSLIP"
    OR = "OR"
    ORCANCEL = "ORCANCEL"
    ORCONFIRM = "ORCONFIRM"
    INQUIRY = "INQUIRY"
def normalize_decimal(value: Union[int, float, Decimal]) -> Decimal:
    """
    Normalize a numeric value so that its decimal part is restricted to
    one of: .00, .25, .50, .75.

    Args:
        value (int | float | Decimal): The numeric value to normalize.

    Returns:
        Decimal: The normalized number with two decimal places.

    Raises:
        TypeError: If the input is not int, float, or Decimal.
    """
    if not isinstance(value, (int, float, Decimal)):
        raise TypeError("Value must be int, float, or Decimal.")

    # Convert to Decimal with 2 decimal precision
    dec_value = Decimal(str(value)).quantize(Decimal("0.00"), rounding=ROUND_HALF_UP)

    integer_part = int(dec_value)
    decimal_part = dec_value - Decimal(integer_part)

    # Possible decimal steps
    steps = [Decimal("0.00"), Decimal("0.25"), Decimal("0.50"), Decimal("0.75")]

    # Find nearest step
    nearest = min(steps, key=lambda x: abs(decimal_part - x))

    # Special case: if rounding pushes above 0.875 → next integer
    if decimal_part >= Decimal("0.875"):
        integer_part += 1
        nearest = Decimal("0.00")

    return Decimal(integer_part) + nearest

class APIRequester:
    def __init__(self, url: str):
        self.url = url

    def send_request(self, xml_data: str) -> Optional[str]:
        """
        ส่ง XML request และ decode response
        :param xml_data: XML string ที่จะส่ง
        :return: ข้อความ decoded หรือ None หากมี error
        """
        try:
            headers = {'Content-Type': 'text/xml'}
            response = requests.post(self.url, headers=headers, data=xml_data.encode("utf-8"))

            # ตรวจสอบ status code
            if response.status_code != 200:
                print(f"[ERROR] Status code: {response.status_code}")
                return None

            # แยกค่า <return>...<return> และ decode base64
            raw_return = response.text.split("<return>")[-1].split("</return>")[0]
            decoded_str = base64.b64decode(raw_return).decode('utf-8')
            
            # log response
            print("[INFO] Response decoded successfully")
            return decoded_str
        
        except Exception as e:
            print(f"[ERROR] Exception: {e}")
            return None
	
class XMLFormatter:
    def __init__(self, funt_data):
        self.funt_data = funt_data
        self.templates = {
            TemplateType.DATAEXCHANGE: self._template_dataexchange,
            TemplateType.CANCEL: self._template_cancel,
            TemplateType.DATAEXCHANGECONFIRM: self._template_dataexchange_confirm,
            TemplateType.REPRINTSLIP: self._template_reprintslip,
            TemplateType.OR: self._template_or,
            TemplateType.ORCANCEL: self._template_or_cancel,
            TemplateType.ORCONFIRM: self._template_or_confirm,
            TemplateType.INQUIRY: self._template_inquiry,
        }

    def safe_format(self, template_func, data: dict) -> str:
        """Format XML template safely with fallback for missing keys"""
        template = template_func()

        class SafeDict(defaultdict):
            def __missing__(self, key):
                return f"{{MISSING:{key}}}"  # หรือ return "" ได้

        try:
            return template.format_map(SafeDict(str, data))
        except Exception as e:
            raise ValueError(f"Template formatting error: {e}")

    def flatten_dict(self, d) -> dict:
        """Flatten nested dict or XML string into flat dict"""
        result = {}

        if isinstance(d, str):
            d = xmltodict.parse(d)

        def _flatten(subdict):
            if isinstance(subdict, dict):
                for k, v in subdict.items():
                    if isinstance(v, (dict, list)):
                        _flatten(v)
                    else:
                        result[k] = v
            elif isinstance(subdict, list):
                for item in subdict:
                    _flatten(item)

        _flatten(d)
        return result

    def build(self, name: TemplateType, debug: bool = False) -> str | None:
        """Build XML envelope for given template"""
        if name not in self.templates:
            return None

        self.funt_data.refresh_date_time()
        data = self.funt_data.get_current()
        if debug:
            pprint(data)

        inner_xml = self.safe_format(self.templates[name], data)

        return f"""<soapenv:Envelope xmlns:soapenv="http://schemas.xmlsoap.org/soap/envelope/" xmlns:por="http://portal.cs/">
  <soapenv:Header/>
  <soapenv:Body>
    <por:CSService>
      <arg0><![CDATA[{inner_xml}]]></arg0>
    </por:CSService>
  </soapenv:Body>
</soapenv:Envelope>"""

    # -----------------------------
    # Template XML
    # -----------------------------
    def _template_dataexchange(self) -> str:
        return  """<?xml version="1.0" encoding="UTF-8"?>
			<HQ_REQUEST>
			<SERVICE_BOX>
			<ADDRESS>
			<VENDOR_ID>{VENDOR_ID}</VENDOR_ID>
			<SERVICE_ID>{SERVICE_ID}</SERVICE_ID>
			<METHOD>DataExchange</METHOD>
			</ADDRESS>
			<DATA>
			<PAYMENT_CHANNEL>{PAYMENT_CHANNEL}</PAYMENT_CHANNEL>
			<VENDOR_ID>{VENDOR_ID}</VENDOR_ID>
			<SERV_ID>{SERVICE_ID}</SERV_ID>
			<SERVICE_ID>{SERVICE_ID}</SERVICE_ID>
			<STORE_ID>{STORE_ID}</STORE_ID>
			<STATION_ID>{MACHINE_ID}</STATION_ID>
			<BUS_DATE>{DATE_BUS}</BUS_DATE>
			<BUS_TIME>{TIME_BUS}</BUS_TIME>
			<SYS_DATE>{DATE_SYS}</SYS_DATE>
			<SYS_TIME>{TIME_SYS}</SYS_TIME>
			<COMMON_TRN_ID>{COMMON_TX_ID}</COMMON_TRN_ID>
			<SEQ_NO>{CLIENT_SERVICE_SEQUENCE}</SEQ_NO>
			<CLIENT_SERV_SEQ>{CLIENT_SEQUENCE_NO}</CLIENT_SERV_SEQ>
			<SHIFT_ID>{SHIFT_ID}</SHIFT_ID>
			<TRANS_TYPE>{TX_TYPE}</TRANS_TYPE>
			<ACCT_NO></ACCT_NO>
			<BILL_AMT>{BILL_AMT}</BILL_AMT>
			<ROUND_BILL_AMT>{BILL_AMT_ROUND}</ROUND_BILL_AMT>
			<VAT_AMT>{BILL_AMT_VAT}</VAT_AMT>
			<REPT_TYPE>{RECEIPT_TYPE}</REPT_TYPE>
			<REPT_NO>{RECEIPT_NO}</REPT_NO>
			<PREV_REF_SEQ></PREV_REF_SEQ>
			<PREV_REF_DATE></PREV_REF_DATE>
			<SERV_CHARGE_NO></SERV_CHARGE_NO>
			<ITEM_NAME>{ITEM_NAME}</ITEM_NAME>
			<ITEM_SELECTION>{ITEM_SELECTIC}</ITEM_SELECTION>
			<EMPLOYEE_ID>{EMPLOYEE_ID}</EMPLOYEE_ID>
			<POS_TAX_ID>{POS_TAX_ID}</POS_TAX_ID>
			<DATA_1>{DATA_1}</DATA_1>
			<DATA_2>{DATA_2}</DATA_2>
			<DATA_3>{DATA_3}</DATA_3>
			<DATA_4>{DATA_4}</DATA_4>
			<DATA_5>{DATA_5}</DATA_5>
			<DATA_6>{DATA_6}</DATA_6>
			<DATA_7>{DATA_7}</DATA_7>
			<DATA_9>{DATA_9}</DATA_9>
			<ZONE>{ZONE}</ZONE>
			<PAYMENT_TYPE>{PAYMENT_TYPE}</PAYMENT_TYPE>
			<CANCEL_ID></CANCEL_ID>
			<CUST_NAME>{CUSTOMER_NAME}</CUST_NAME>
			<CUST_ADDR_1>{CUSTOMER_ADDR_1}</CUST_ADDR_1>
			<CUST_ADDR_2>{CUSTOMER_ADDR_2}</CUST_ADDR_2>
			<CUST_ADDR_3>{CUSTOMER_ADDR_3}</CUST_ADDR_3>
			<CUST_PHONE_NO>{CUSTOMER_TEL_NO}</CUST_PHONE_NO>
			</DATA>
			</SERVICE_BOX>
			</HQ_REQUEST>
			"""
    def _template_cancel(self) -> str:
        return  """<?xml version="1.0" encoding="UTF-8"?>
          <HQ_REQUEST>
            <SERVICE_BOX>
              <ADDRESS>
                <VENDOR_ID>{VENDOR_ID}</VENDOR_ID>
                <SERVICE_ID>{SERVICE_ID}</SERVICE_ID>
                <METHOD>Cancel</METHOD>
              </ADDRESS>
              <DATA>
                <PAYMENT_CHANNEL>{PAYMENT_CHANNEL}</PAYMENT_CHANNEL>
                <VENDOR_ID>{VENDOR_ID_OUT}</VENDOR_ID>
                <SERV_ID>{SERVICE_ID_OUT}</SERV_ID>
                <SERVICE_ID>{SERVICE_ID_OUT}</SERVICE_ID>
                <STORE_ID>{STORE_ID}</STORE_ID>
                <STATION_ID>{MACHINE_ID}</STATION_ID>
                <BUS_DATE>{DATE_BUS}</BUS_DATE>
                <BUS_TIME>{TIME_BUS}</BUS_TIME>
                <TX_ID>{TX_ID}</TX_ID>
                <PAYMENT_TYPE>{PAYMENT_TYPE}</PAYMENT_TYPE>
                <CANCEL_ID></CANCEL_ID>
              </DATA>
            </SERVICE_BOX>
          </HQ_REQUEST>"""
    def _template_dataexchange_confirm(self) -> str:
        return """<?xml version="1.0" encoding="UTF-8"?>
<HQ_REQUEST>
<SERVICE_BOX>
<ADDRESS>
<VENDOR_ID>{VENDOR_ID}</VENDOR_ID>
<SERVICE_ID>{SERVICE_ID}</SERVICE_ID>
<METHOD>DataExchangeConfirm</METHOD>
</ADDRESS>
<DATA>
<PAYMENT_CHANNEL>{PAYMENT_CHANNEL}</PAYMENT_CHANNEL>
<VENDOR_ID>{VENDOR_ID_OUT}</VENDOR_ID>
<SERV_ID>{SERVICE_ID_OUT}</SERV_ID>
<SERVICE_ID>{SERVICE_ID_OUT}</SERVICE_ID>
<STATION_ID>{MACHINE_ID}</STATION_ID>
<STORE_ID>{STORE_ID}</STORE_ID>
<BUS_DATE>{DATE_BUS}</BUS_DATE>
<BUS_TIME>{TIME_BUS}</BUS_TIME>
<SYS_DATE>{DATE_SYS}</SYS_DATE>
<SYS_TIME>{TIME_SYS}</SYS_TIME>
<TX_ID>{TX_ID}</TX_ID>
<SEQ_NO>{SEQ_NO}</SEQ_NO>
<EMPLOYEE_ID>{EMPLOYEE_ID}</EMPLOYEE_ID>
<CLIENT_SERV_SEQ>{CLIENT_SERV_SEQ}</CLIENT_SERV_SEQ>
<SERV_ID>{SERVICE_ID}</SERV_ID>
<BILL_AMT>{BILL_AMT}</BILL_AMT>
<ROUND_BILL_AMT>{ROUND_BILL_AMT}</ROUND_BILL_AMT>
<ACCT_NO></ACCT_NO>
<VAT_AMT>{VAT_AMT}</VAT_AMT>
<DATA_1>{DATA_1_NO}</DATA_1>
<DATA_2>{DATA_2_NO}</DATA_2>
<DATA_3>{DATA_3_NO}</DATA_3>
<DATA_4>{DATA_4_NO}</DATA_4>
<DATA_5>{DATA_5_NO}</DATA_5>
<DATA_6>{DATA_6_NO}</DATA_6>
<DATA_7>{DATA_7_NO}</DATA_7>
<DATA_9>{DATA_9_NO}</DATA_9>
<ZONE>{ZONE}</ZONE>
<PAYMENT_TYPE>001</PAYMENT_TYPE>
<TOT_BILL_TRANS></TOT_BILL_TRANS>
<TOT_BILL_AMT></TOT_BILL_AMT>
<TOT_VENDOR_TRANS></TOT_VENDOR_TRANS>
<TOT_VENDOR_AMT></TOT_VENDOR_AMT>
<TOT_COUNTER_TRANS></TOT_COUNTER_TRANS>
<TOT_COUNTER_AMT></TOT_COUNTER_AMT>
<TOT_CLIENT_TRANS></TOT_CLIENT_TRANS>
<TOT_CLIENT_AMT></TOT_CLIENT_AMT>
<TOT_BILL_TRANS_OR></TOT_BILL_TRANS_OR>
<TOT_BILL_AMT_OR></TOT_BILL_AMT_OR>
<CANCEL_ID></CANCEL_ID>
<CUST_NAME>{CUST_NAME}</CUST_NAME>
<CUST_ADDR_1>{CUST_ADDR_1}</CUST_ADDR_1>
<CUST_ADDR_2>{CUST_ADDR_2}</CUST_ADDR_2>
<CUST_ADDR_3>{CUST_ADDR_3}</CUST_ADDR_3>
<CUST_PHONE_NO>{CUST_PHONE_NO}</CUST_PHONE_NO>
</DATA>
</SERVICE_BOX>
</HQ_REQUEST>"""
    def _template_reprintslip(self) -> str:
        return """<?xml version="1.0" encoding="UTF-8"?>
<HQ_REQUEST>
<SERVICE_BOX>
<ADDRESS>
<VENDOR_ID>{VENDOR_ID}</VENDOR_ID>
<SERVICE_ID>{SERVICE_ID}</SERVICE_ID>
<METHOD>REPRINTSLIP</METHOD>
</ADDRESS>
<DATA>
<PAYMENT_CHANNEL>{PAYMENT_CHANNEL}</PAYMENT_CHANNEL>
<VENDOR_ID>{VENDOR_ID_OUT}</VENDOR_ID>
<SERV_ID>{SERVICE_ID_OUT}</SERV_ID>
<SERVICE_ID>{SERVICE_ID_OUT}</SERVICE_ID>
<STORE_ID>{STORE_ID}</STORE_ID>
<STATION_ID>{MACHINE_ID}</STATION_ID>
<BUS_DATE>{DATE_BUS}</BUS_DATE>
<BUS_TIME>{TIME_BUS}</BUS_TIME>
<COMMON_TRN_ID>{COMMON_TX_ID}</COMMON_TRN_ID>
<SEQ_NO>{SEQ_NO}</SEQ_NO>
<CLIENT_SERV_SEQ>{CLIENT_SERV_SEQ}</CLIENT_SERV_SEQ>
<SHIFT_ID>{SHIFT_ID}</SHIFT_ID>
<TRANS_TYPE>{TX_TYPE}</TRANS_TYPE>
<ACCT_NO></ACCT_NO>
<BILL_AMT>{BILL_AMT}</BILL_AMT>
<ROUND_BILL_AMT>{ROUND_BILL_AMT}</ROUND_BILL_AMT>
<VAT_AMT>{VAT_AMT}</VAT_AMT>
<REPT_TYPE>{REPT_TYPE}</REPT_TYPE>
<TX_ID>{TX_ID}</TX_ID>
<REPT_NO></REPT_NO>
<PREV_REF_SEQ></PREV_REF_SEQ>
<PREV_REF_DATE></PREV_REF_DATE>
<SERV_CHARGE_NO></SERV_CHARGE_NO>
<ITEM_NAME>{ITEM_NAME}</ITEM_NAME>
<ITEM_SELECTION>{ITEM_SELECTIC}</ITEM_SELECTION>
<EMPLOYEE_ID>{EMPLOYEE_ID}</EMPLOYEE_ID>
<POS_TAX_ID>{POS_TAX_ID}</POS_TAX_ID>
<DATA_1>{DATA_1_NO}</DATA_1>
<DATA_2>{DATA_2_NO}</DATA_2>
<DATA_3>{DATA_3_NO}</DATA_3>
<DATA_4>{DATA_4_NO}</DATA_4>
<DATA_5>{DATA_5_NO}</DATA_5>
<DATA_6>{DATA_6_NO}</DATA_6>
<DATA_7>{DATA_7_NO}</DATA_7>
<DATA_9>{DATA_9_NO}</DATA_9>
<ZONE>{ZONE}</ZONE>
<CANCEL_ID></CANCEL_ID>
</DATA>
</SERVICE_BOX>
</HQ_REQUEST>"""
    def _template_or(self) -> str:
        return """<?xml version="1.0" encoding="UTF-8"?>
<HQ_REQUEST>
<SERVICE_BOX>
<ADDRESS>
<VENDOR_ID>{VENDOR_ID}</VENDOR_ID>
<SERVICE_ID>{SERVICE_ID}</SERVICE_ID>
<METHOD>OR</METHOD>
</ADDRESS>
<DATA>
<PAYMENT_CHANNEL>{PAYMENT_CHANNEL}</PAYMENT_CHANNEL>
<VENDOR_ID>{VENDOR_ID_OUT}</VENDOR_ID>
<SERVICE_ID>{SERVICE_ID_OUT}</SERVICE_ID>
<SERV_ID>{SERVICE_ID_OUT}</SERV_ID>
<STORE_ID>{STORE_ID}</STORE_ID>
<STATION_ID>{MACHINE_ID}</STATION_ID>
<BUS_DATE>{DATE_BUS}</BUS_DATE>
<BUS_TIME>{TIME_BUS}</BUS_TIME>
<SYS_DATE>{DATE_SYS}</SYS_DATE>
<SYS_TIME>{TIME_SYS}</SYS_TIME>
<TX_ID>{TX_ID}</TX_ID>
<BILL_AMT>{BILL_AMT}</BILL_AMT>
<ROUND_BILL_AMT>{ROUND_BILL_AMT}</ROUND_BILL_AMT>
<VAT_AMT>{VAT_AMT}</VAT_AMT>
<PAYMENT_TYPE>001</PAYMENT_TYPE>
<CANCEL_ID></CANCEL_ID>
</DATA>
</SERVICE_BOX>
</HQ_REQUEST>"""
    def _template_or_cancel(self) -> str:
        return """<?xml version="1.0" encoding="UTF-8"?>
<HQ_REQUEST>
<SERVICE_BOX>
<ADDRESS>
<VENDOR_ID>{VENDOR_ID}</VENDOR_ID>
<SERVICE_ID>{SERVICE_ID}</SERVICE_ID>
<METHOD>ORCancel</METHOD>
</ADDRESS>
<DATA>
<PAYMENT_CHANNEL>{PAYMENT_CHANNEL}</PAYMENT_CHANNEL>
<VENDOR_ID>{VENDOR_ID_OUT}</VENDOR_ID>
<SERVICE_ID>{SERVICE_ID_OUT}</SERVICE_ID>
<SERV_ID>{SERVICE_ID_OUT}</SERV_ID>
<STORE_ID>{STORE_ID}</STORE_ID>
<STATION_ID>{MACHINE_ID}</STATION_ID>
<BUS_DATE>{DATE_BUS}</BUS_DATE>
<BUS_TIME>{TIME_BUS}</BUS_TIME>
<TX_ID>{TX_ID}</TX_ID>
<PAYMENT_TYPE>001</PAYMENT_TYPE>
<CANCEL_ID></CANCEL_ID>
</DATA>
</SERVICE_BOX>
</HQ_REQUEST>"""
    def _template_or_confirm(self) -> str:
        return """<?xml version="1.0" encoding="UTF-8"?>
<HQ_REQUEST>
<SERVICE_BOX>
<ADDRESS>
<VENDOR_ID>{VENDOR_ID}</VENDOR_ID>
<SERVICE_ID>{SERVICE_ID}</SERVICE_ID>
<METHOD>ORConfirm</METHOD>
</ADDRESS>
<DATA>
<PAYMENT_CHANNEL>{PAYMENT_CHANNEL}</PAYMENT_CHANNEL>
<VENDOR_ID>{VENDOR_ID_OUT}</VENDOR_ID>
<SERVICE_ID>{SERVICE_ID_OUT}</SERVICE_ID>
<SERV_ID>{SERVICE_ID_OUT}</SERV_ID>
<STORE_ID>{STORE_ID}</STORE_ID>
<STATION_ID>{MACHINE_ID}</STATION_ID>
<BUS_DATE>{DATE_BUS}</BUS_DATE>
<BUS_TIME>{TIME_BUS}</BUS_TIME>
<BILL_AMT>{BILL_AMT}</BILL_AMT>
<ROUND_BILL_AMT>{ROUND_BILL_AMT}</ROUND_BILL_AMT>
<VAT_AMT>{VAT_AMT}</VAT_AMT>
<TX_ID>{TX_ID}</TX_ID>
<SEQ_NO>{SEQ_NO}</SEQ_NO>
<CLIENT_SERV_SEQ>{CLIENT_SERV_SEQ}</CLIENT_SERV_SEQ>
<SERV_ID>{SERVICE_ID}</SERV_ID>
<DATA_1>{DATA_1_NO}</DATA_1>
<DATA_2>{DATA_2_NO}</DATA_2>
<DATA_3>{DATA_3_NO}</DATA_3>
<DATA_4>{DATA_4_NO}</DATA_4>
<DATA_5>{DATA_5_NO}</DATA_5>
<DATA_6>{DATA_6_NO}</DATA_6>
<DATA_7>{DATA_7_NO}</DATA_7>
<DATA_9>{DATA_9_NO}</DATA_9>
<ZONE>{ZONE}</ZONE>
<PAYMENT_TYPE>001</PAYMENT_TYPE>
<TOT_BILL_TRANS></TOT_BILL_TRANS>
<TOT_BILL_AMT></TOT_BILL_AMT>
<TOT_VENDOR_TRANS></TOT_VENDOR_TRANS>
<TOT_VENDOR_AMT></TOT_VENDOR_AMT>
<TOT_COUNTER_TRANS></TOT_COUNTER_TRANS>
<TOT_COUNTER_AMT></TOT_COUNTER_AMT>
<TOT_CLIENT_TRANS></TOT_CLIENT_TRANS>
<TOT_CLIENT_AMT></TOT_CLIENT_AMT>
<TOT_BILL_TRANS_OR></TOT_BILL_TRANS_OR>
<TOT_BILL_AMT_OR></TOT_BILL_AMT_OR>
<CANCEL_ID></CANCEL_ID>
</DATA>
</SERVICE_BOX>
</HQ_REQUEST>"""
    def _template_inquiry(self) -> str:
        return """<?xml version="1.0" encoding="UTF-8"?>
<HQ_REQUEST>
<SERVICE_BOX>
<ADDRESS>
<VENDOR_ID>{VENDOR_ID}</VENDOR_ID>
<SERVICE_ID>{SERVICE_ID}</SERVICE_ID>
<METHOD>Inquiry</METHOD>
</ADDRESS>
<DATA>
<PAYMENT_CHANNEL>{PAYMENT_CHANNEL}</PAYMENT_CHANNEL>
<VENDOR_ID>{VENDOR_ID}</VENDOR_ID>
<SERV_ID>{SERVICE_ID}</SERV_ID>
<SERVICE_ID>{SERVICE_ID}</SERVICE_ID>
<STORE_ID>{STORE_ID}</STORE_ID>
<STATION_ID>{MACHINE_ID}</STATION_ID>
<BUS_DATE>{DATE_BUS}</BUS_DATE>
<BUS_TIME>{TIME_BUS}</BUS_TIME>
<SYS_DATE>{DATE_SYS}</SYS_DATE>
<SYS_TIME>{TIME_SYS}</SYS_TIME>
<COMMON_TRN_ID>{COMMON_TX_ID}</COMMON_TRN_ID>
<SEQ_NO>{SEQ_NO}</SEQ_NO>
<CLIENT_SERV_SEQ>{CLIENT_SERV_SEQ}</CLIENT_SERV_SEQ>
<SHIFT_ID>{SHIFT_ID}</SHIFT_ID>
<TRANS_TYPE>{TX_TYPE}</TRANS_TYPE>
<ACCT_NO></ACCT_NO>
<BILL_AMT>{BILL_AMT}</BILL_AMT>
<ROUND_BILL_AMT>{ROUND_BILL_AMT}</ROUND_BILL_AMT>
<VAT_AMT>{VAT_AMT}</VAT_AMT>
<REPT_TYPE>{REPT_TYPE}</REPT_TYPE>
<REPT_NO></REPT_NO>
<PREV_REF_SEQ></PREV_REF_SEQ>
<PREV_REF_DATE></PREV_REF_DATE>
<SERV_CHARGE_NO></SERV_CHARGE_NO>
<ITEM_NAME>{ITEM_NAME}</ITEM_NAME>
<ITEM_SELECTION>{ITEM_SELECTIC}</ITEM_SELECTION>
<EMPLOYEE_ID>{EMPLOYEE_ID}</EMPLOYEE_ID>
<POS_TAX_ID>{POS_TAX_ID}</POS_TAX_ID>
<DATA_1>{DATA_1_NO}</DATA_1>
<DATA_2>{DATA_2_NO}</DATA_2>
<DATA_3>{DATA_3_NO}</DATA_3>
<DATA_4>{DATA_4_NO}</DATA_4>
<DATA_5>{DATA_5_NO}</DATA_5>
<DATA_6>{DATA_6_NO}</DATA_6>
<DATA_7>{DATA_7_NO}</DATA_7>
<DATA_9>{DATA_9_NO}</DATA_9>
<ZONE>{ZONE}</ZONE>
<PAYMENT_TYPE>{PAYMENT_TYPE}</PAYMENT_TYPE>
<CANCEL_ID></CANCEL_ID>
</DATA>
</SERVICE_BOX>
</HQ_REQUEST>"""

class DataStore:
    def __init__(self,actions_round: int =2 ):
        self.fake = Faker(THAI)
        self._saved_data = {}   # เก็บข้อมูลที่ save แล้ว
        self._save_index = ZERO    # running index
        self._history = []   # เก็บ state เก่า (undo)
        self._future = []    # เก็บ state หลัง undo (redo)
        self._current_data = {}
        self._history_all = {}
        self.actions_per_round = actions_round
        
        # กำหนดค่าเริ่มต้น
        self._current_data = {
            "FUNTION": "INTIALI",
            "SUCCESS":UNKNOWN,
	     "CODE":UNKNOWN,
	     "DESCRIPTOR":UNKNOWN,
            "STORE_ID": "09884",
            "MACHINE_ID": ''.join(random.choices("1234", k=1)),
            "VENDOR_CODE": UNKNOWN,
            "SERVICE_ID": UNKNOWN,
            "VENDOR_ID": UNKNOWN,
            "ITEM_NAME": UNKNOWN,
            "COMMON_TX_ID": str(randint(ZERO, 100)),
            "SHIFT_ID": ''.join(random.choices("123456789", k=1)),
            "TX_TYPE": N,
            "ITEM_SELECTIC": N,
            "PAYMENT_TYPE": CASH,
            "ZONE": 1,
            "EMPLOYEE_ID": ''.join(random.choices("056789", k=7)),
            "POS_TAX_ID": ''.join(random.choices("0123456789", k=13)),
            "RECEIPT_NO": str(randint(ZERO, 1000)),
            "RECEIPT_TYPE": H,
            "PAYMENT_CHANNEL": POS,
	     "PRINTSLIP":UNKNOWN,


            "SERV_ID": UNKNOWN,
            "TX_ID": UNKNOWN,
            "CLIENT_SERVICE_SEQUENCE": UNKNOWN,
            "CLIENT_SEQUENCE_NO": UNKNOWN,
            "DATA_1_NO": UNKNOWN,
            "DATA_2_NO": UNKNOWN,
            "DATA_3_NO": UNKNOWN,
            "DATA_4_NO": UNKNOWN,
            "DATA_5_NO": UNKNOWN,
            "DATA_6_NO": UNKNOWN,
            "DATA_7_NO": UNKNOWN,
            "DATA_9_NO": UNKNOWN,
            "DATA_1": UNKNOWN,
            "DATA_2": UNKNOWN,
            "DATA_3": UNKNOWN,
            "DATA_4": UNKNOWN,
            "DATA_5": UNKNOWN,
            "DATA_6": UNKNOWN,
            "DATA_7": UNKNOWN,
            "DATA_9": UNKNOWN,
            
	     "BILL_AMT_ABOVE_MAX": 90001,
            "BILL_AMT_MAX": 90000,
            "BILL_AMT_CENTER": 49000,
            "BILL_AMT_MIN": 1,
	     "BILL_AMT_BELOW_MIN": ZERO,
	     "BILL_AMT_NULL": "",
            "BILL_AMT_ZERO": ZERO,
            "BILL_AMT_DECIMAL": ZERO,
            "BILL_AMT_NON_DECIMAL": ZERO,
            "BILL_AMT_EDIT": ZERO,
            "BILL_AMT": ZERO,
            "BILL_AMT_ROUND": ZERO,
            "BILL_AMT_VAT": ZERO,
	     "FEE":UNKNOWN,
	     "FEE_VAT":UNKNOWN,
         
            "CUSTOMER_NAME": self.fake.first_name(),
            "CUSTOMER_ADDR_1": self.fake.address(),
            "CUSTOMER_ADDR_2": self.fake.address(),
            "CUSTOMER_ADDR_3": UNKNOWN,
            "CUSTOMER_TEL_NO": self.fake.phone_number(),
            "CUSTOMER_TAX_ID": UNKNOWN,
	     "CUSTOMER_BRANCH_CODE":UNKNOWN,
	     "CUSTOMER_RECEIPT_NAME":UNKNOWN,
	     "CUSTOMER_RECEIPT_ADDR":UNKNOWN,

            "STEP": ZERO,
            "HOURS": ZERO,
            "MINUTES": ZERO,
            "HOURS_NEW": ZERO,
            "MINUTES_NEW": ZERO,
            "ACCT_NO":"",
            "DATE_SYS": datetime.now().strftime("%Y/%m/%d"),
            "TIME_SYS": datetime.now().strftime("%X"),
            "DATE_BUS": datetime.now().strftime("%Y/%m/%d"),
            "TIME_BUS": datetime.now().strftime("%X"),
            


	 }
        self.refresh_bill_amt()
        self.refresh_value

    # ------------------------------
    # Function จัดการข้อมูล
    # ------------------------------

    def Generator_DATA(self,value):
        """แก้ไขค่าปัจจุบัน"""
        if isinstance(value, int):
            return ''.join(random.choices("0123456789", k=value))
        elif isinstance(value, str):
            return value
        elif isinstance(value, list):
            return value[0] if value else None
        else:
            return None
    def add_field(self, key, value):
        """เพิ่มตัวแปรใหม่"""
        print(f"{key} ทำการสร้างตัวเเปรใหม่")
        self._current_data[key] = value
    def refresh_date_time(self):
        """แก้ไขค่าปัจจุบัน"""
        self._current_data[DATE_SYS] = datetime.now().strftime("%Y/%m/%d")
        self._current_data[TIME_SYS] = datetime.now().strftime("%X")
        self._current_data[DATE_BUS] = datetime.now().strftime("%Y/%m/%d")
        self._current_data[TIME_BUS] = datetime.now().strftime("%X")
    def refresh_bill_amt(self):
        """แก้ไขค่าปัจจุบัน"""
        self._current_data["BILL_AMT_ABOVE_MAX"] = self._current_data["BILL_AMT_MAX"]+1
        self._current_data["BILL_AMT_BELOW_MIN"] = self._current_data["BILL_AMT_MIN"]-0.5
        self._current_data["BILL_AMT_EDIT"] = self._current_data["BILL_AMT"]+random.choice([x for x in range(1, 99) if x % 10 == 0])
        self._current_data["BILL_AMT_ROUND"] = normalize_decimal(self._current_data["BILL_AMT"])
        self._current_data["BILL_AMT_DECIMAL"] = self._current_data.get("BILL_AMT")+(random.choice([x for x in range(1, 99) if x % 25 == 0])/100)
        self._current_data["BILL_AMT_NON_DECIMAL"] = self._current_data["BILL_AMT"]+(random.choice([x for x in range(1, 99) if x % 25 != 0])/100)
    def refresh_value(self):
        """แก้ไขค่าปัจจุบัน"""
        key_value = ["DATA_1","DATA_2","DATA_3","DATA_4","DATA_5","DATA_6"]
        key = ["DATA_1_NO","DATA_2_NO","DATA_3_NO","DATA_4_NO","DATA_5_NO","DATA_6_NO"]
        for k,kv in zip(key,key_value):
           if k in self._current_data:
               self._current_data[kv] = self.Generator_DATA(self._current_data[k])
           else:
               raise KeyError(f"{key} ไม่พบใน current data")

    
    def get(self, key, default=None):
        """ดึงค่าตาม key"""
        return self._current_data.get(key, default)

    def get_current(self):
        """ดึงค่าทั้งหมด"""
        return copy.deepcopy(self._current_data)
    def get_save_value(self, key=None):
        """ดึงค่าที่ save แล้ว"""
        if key:
            return self._saved_data.get(key, None)
        return copy.deepcopy(self._saved_data)   
    def get_all_value(self, index=None):
        """ดึงค่าที่ save แล้ว"""
        if index:
            return self._history_all.get(index, None)
        return copy.deepcopy(self._history_all)
    def roll_back_value(self,index):
        self._current_data = copy.deepcopy(self._history_all).get(index,None).get(REQUEST,None)
        
    def undo(self):
        """ย้อนกลับ 1 step"""
        if not self._history:
            ic("ไม่มีข้อมูลให้ undo")
            return
        self._future.append(copy.deepcopy(self._current_data))
        self._current_data = self._history.pop()

    def redo(self):
        """ทำซ้ำ (กลับไปสถานะถัดไป หลัง undo)"""
        if not self._future:
            print("ไม่มีข้อมูลให้ redo")
            return
        self._history.append(copy.deepcopy(self._current_data))
        self._current_data = self._future.pop()

    def update(self, key, value):
        """แก้ไขค่าปัจจุบัน"""
        self._save_history()
        if key in self._current_data:
            self._current_data[key] = value
        else:
            self.add_field(key,value)
        
    def _save_history(self):
        """บันทึกสถานะก่อนแก้ไข"""
        self._history.append(copy.deepcopy(self._current_data))
        self._future.clear()  # reset redo ทุกครั้งที่มีการแก้ไขใหม่
        
    def save(self,action=None,key=False):
        """บันทึก snapshot (แก้ไขไม่ได้)"""
        if not action:
            raise ValueError("ต้องใส่ชื่อ action ด้วย")
        self._saved_data[action] = copy.deepcopy(self._current_data)
        if len(self._saved_data) >= self.actions_per_round or key:
            self._save_index += 1
            self._history_all[self._save_index] = copy.deepcopy(self._saved_data)
            self._saved_data = {}
            ic(f"""บันทึก snapshot (index : {self._save_index})""")
            self.current_to_dataframe()
    
    def all_to_dataframe(self):
        rows = []
        all_keys = set()
        for idx, actions in self._history_all.items():
            for action_name, values in actions.items():
                row = {"INDEXS": idx, "ACTIONS": action_name}
                row.update(values)
                rows.append(row)
                all_keys.update(values.keys())
        columns = ["INDEXS", "ACTIONS"].append(all_keys) #sorted(all_keys)
        df = pd.DataFrame(rows, columns=columns)
        show(df.fillna(""))

    def current_to_dataframe(self):
        """
        แปลง _history_all[_save_index] เป็น DataFrame
        - รองรับ list → explode row แบบ pair-wise
        - เติมค่า "" แทน None
        - สร้างตัวแปรใหม่ตาม field ที่กำหนด
        """
        idx = self._save_index
        if idx == 0 or idx not in self._history_all:
            print("no data")
            return pd.DataFrame()
    
        history = self._history_all[idx]
        rows = []
    
        for action_name, values in history.items():
            base_row = {"INDEXS": idx, "ACTIONS": action_name}
    
            # หา list fields
            list_fields = {k: v for k, v in values.items() if isinstance(v, list)}
            max_len = max((len(v) for v in list_fields.values()), default=1)
    
            for i in range(max_len):
                row = base_row.copy()
                for key, val in values.items():
                    if isinstance(val, list):
                        # ใช้ค่าตาม index i หรือ "" ถ้าไม่มี
                        row[key] = val[i] if i < len(val) else ""
                    else:
                        row[key] = val if val is not None else ""
    
                # สร้างตัวแปรใหม่
                for f in ["CLIENT_NO","CLIENT_SEQUENCE","CLIENT_SERVICE_SEQUENCE_OR","CLIENT_SEQUENCE_NO_OR"]:
                    if f in values:
                        row[f"{f}_NEW"] = values[f]
    
                rows.append(row)
    
        # รวมคอลัมน์ทั้งหมด
        all_keys = set()
        for row in rows:
            all_keys.update(row.keys())
    
        columns = ["INDEXS", "ACTIONS"] + sorted(k for k in all_keys if k not in ["INDEXS", "ACTIONS"])
        df = pd.DataFrame(rows, columns=columns)
        show(df.fillna("")) 

    
def send_api(api:APIRequester,method:TemplateType,data:DataStore):
    formatter = XMLFormatter(data)
    xml_str = formatter.build(method)
    data.update(FUNTION,method)
    data.save(REQUEST)
    decoded_response = formatter.flatten_dict(api.send_request(xml_str))
	# pprint(decoded_response)
    for key, value in decoded_response.items():
        if value is not None :
           left, right =split_by_pipe(value)
           data.update(key,[left, right] if right is not None else left)
#     clean_none = lambda d: {k: clean_none(v) if isinstance(v, dict) else v for k, v in d.items() if v is not None}

# อัปเดต data ด้วย decoded_response แบบกรองค่า None
#     data.update(clean_none(decoded_response))

#     ic(decoded_response)
#     ic(data.get_current())
    if check_status(decoded_response):
        prefix = cut_prefix(data._current_data[TX_ID],5)
        data.update(CLIENT_SEQUENCE_NO,prefix.get(DATAEXCHANGE,""))
        data.update(CLIENT_SERVICE_SEQUENCE,prefix.get(OR,""))
        data.save(RESPONSE)
    else:
        data.save(ERROR)
    return data
   
def split_by_pipe(value: str) -> tuple[str, str]:
    if "|" not in value:
        return value, None 
    left, right = value.split("|", 1) 
    return left, right

def check_status(value: dict) -> bool:
    return value.get("CODE","801999") == "100"

def cut_prefix(values: list[str], n: int = 5) -> dict:
    add = sum(1 for v in values if v is not None)
    ic(values)
    return {DATAEXCHANGE:[str(int(v) + add).zfill(n)[-n:]  for v in values if v is not None],
            OR:[str(int(v)).zfill(n)[:n]  for v in values if v is not None ]}





In [ ]:
URL = "http://testcspos.counterservice.co.th:8001/DCWSCDSONLINE/WSCDSService"
API = APIRequester(URL)
APP = DataStore()
APP.update(DATA_1,"11270110126")
APP.update(DATA_3,10)
# APP.update(DATA_3,10)
APP.update(STORE_ID,"09884")
APP.update(BILL_AMT,80)
APP.update(BILL_AMT_MIN,1)
APP.update(BILL_AMT_MAX,49000)
APP.update(VENDOR_CODE,"10792")
APP.update(VENDOR_ID,"0994000164904")
APP.update(SERVICE_ID,"00")
APP.refresh_value()
APP.save("INTIALI",True)


ic| f"""บันทึก snapshot (index : {self._save_index})""": 'บันทึก snapshot (index : 1)'


Loading ITables v2.5.2 from the internet... (need help?)


In [ ]:
APP = send_api(api=API,data=APP,method=DATAEXCHANGE)


ic| f"""บันทึก snapshot (index : {self._save_index})""": 'บันทึก snapshot (index : 2)'


[INFO] Response decoded successfully


Loading ITables v2.5.2 from the internet... (need help?)


In [ ]:
APP = send_api(api=API,data=APP,method=CANCEL)


ic| f"""บันทึก snapshot (index : {self._save_index})""": 'บันทึก snapshot (index : 3)'


[INFO] Response decoded successfully


Loading ITables v2.5.2 from the internet... (need help?)


In [ ]:
# --- patch: แก้ safe_format, flatten_dict, DataStore init call, send_api, cut_prefix ---

# เอา duplicate import ออก (ถ้ามี)
# import xmltodict  # ถ้ามี 1 ครั้งก็พอ

from collections import defaultdict
import xmltodict

class XMLFormatter:
    ...
    def safe_format(self, template_func, data: dict) -> str:
        """Format XML template safely with fallback for missing keys"""
        template = template_func()

        class SafeDict(dict):
            def __missing__(self, key):
                # ถ้าต้องการเห็น key ที่หาย ให้เปลี่ยนเป็น f"{{MISSING:{key}}}"
                return ""

        # สร้าง mapping โดยใช้ข้อมูลจาก data
        mapping = SafeDict()
        if data:
            # copy แต่ให้แน่ใจเป็น string เพื่อ format ไม่พัง
            for k, v in data.items():
                # ถ้าเป็น None ให้เป็น ""
                mapping[str(k)] = "" if v is None else str(v)

        try:
            return template.format_map(mapping)
        except Exception as e:
            raise ValueError(f"Template formatting error: {e}")

    def flatten_dict(self, d) -> dict:
        """Flatten nested dict or XML string into flat dict with underscore-separated keys.
        If input is XML string, parse with xmltodict first.
        """
        result = {}

        if d is None:
            return result

        if isinstance(d, str):
            # ถ้าเป็น XML string หรือ JSON-like string ให้ parse xmltodict
            try:
                parsed = xmltodict.parse(d)
            except Exception:
                # ไม่ใช่ XML -> คืนค่า dict ว่าง
                return {}
        elif isinstance(d, dict):
            parsed = d
        else:
            # อื่นๆ เช่น object type -> คืนว่าง
            return {}

        def _flatten(obj, prefix=""):
            if isinstance(obj, dict):
                for k, v in obj.items():
                    key = f"{prefix}_{k}" if prefix else str(k)
                    _flatten(v, key)
            elif isinstance(obj, list):
                # เก็บรายการเป็น index-suffixed keys หรือ join เป็น comma
                # ในที่นี้จะสร้าง key_0, key_1,... เพื่อความชัดเจน
                for i, item in enumerate(obj):
                    _flatten(item, f"{prefix}_{i}")
            else:
                # leaf node
                # แปลงค่าทุกค่าเป็น string (หรือเก็บ None เป็น "")
                result[prefix] = "" if obj is None else str(obj)

        _flatten(parsed)
        return result

    ...

class DataStore:
    def __init__(self, actions_round: int = 2):
        ...
        # หลังตั้ง _current_data เรียบร้อย ให้เรียกทั้งสองฟังก์ชันจริงๆ
        self.refresh_bill_amt()
        self.refresh_value()   # <-- ใส่ () ให้เรียกจริง

    ...

def send_api(api: APIRequester, method: TemplateType, data: DataStore):
    formatter = XMLFormatter(data)
    xml_str = formatter.build(method)
    # บันทึกฟังก์ชันที่เรียก
    data.update(FUNTION, method)
    data.save(REQUEST)

    # เรียก API และตรวจสอบ None
    raw = api.send_request(xml_str)
    if not raw:
        print("[ERROR] No response or failed to fetch response")
        data.save(ERROR)
        return data

    # raw ควรเป็น XML string ที่มีข้อมูล base64-decoded -> flatten เป็น dict
    decoded_response = formatter.flatten_dict(raw)

    # ถ้า flatten ไม่ได้ return dict หรือว่าง ให้บันทึกและ return
    if not decoded_response:
        print("[WARN] decoded_response is empty")
        data.save(ERROR)
        return data

    # อัปเดต data จาก decoded_response
    for key, value in decoded_response.items():
        if value is not None:
            # value ควรเป็น string; ถ้ามี '|' แยกซ้าย/ขวา
            left, right = split_by_pipe(value)
            # ถ้ามี right ให้เก็บเป็น list [left, right] เพื่อให้ตามโค้ดเดิม
            data.update(key, [left, right] if right is not None else left)

    # ตรวจสอบ status
    if check_status(decoded_response):
        # เตรียม prefix — อาจได้ค่าเป็น str หรือ list -> ให้ cut_prefix รองรับทั้งสอง
        tx_val = data._current_data.get(TX_ID, "")
        prefix = cut_prefix(tx_val, 5)
        # ถ้ามี prefix จะได้ dict กับคีย์ DATAEXCHANGE และ OR
        data.update(CLIENT_SEQUENCE_NO, prefix.get(DATAEXCHANGE, ""))
        data.update(CLIENT_SERVICE_SEQUENCE, prefix.get(OR, ""))
        data.save(RESPONSE)
    else:
        data.save(ERROR)
    return data

def cut_prefix(values, n: int = 5) -> dict:
    """
    รับ `values` เป็น:
      - list[str] ของตัวเลข
      - str: ถ้าเป็น comma-separated จะ split, ถ้าเป็นตัวเลขเดียวก็แปลงเป็น list
    คืน dict ที่มีสองคีย์ DATAEXCHANGE และ OR (แต่ถ้าแปลงไม่ได้จะคืนค่าว่าง)
    """
    # normalize เป็น list[str]
    vals = []
    if values is None:
        vals = []
    elif isinstance(values, list):
        vals = [str(v) for v in values if v is not None and str(v).strip() != ""]
    elif isinstance(values, str):
        s = values.strip()
        # ถ้าเป็น comma separated
        if "," in s:
            vals = [p.strip() for p in s.split(",") if p.strip() != ""]
        elif s == "":
            vals = []
        else:
            vals = [s]
    else:
        # กรณีอื่นๆ พยายามแปลงเป็น str
        vals = [str(values)]

    # filter ให้เป็นตัวเลขเท่านั้น (ป้องกัน ValueError)
    numeric_vals = []
    for v in vals:
        v_clean = v.strip()
        if v_clean.isdigit():
            numeric_vals.append(v_clean)
        else:
            # ลองแยก non-digit ตัวหน้าสุดออก (เช่น "00012") หรือข้ามไป
            filtered = ''.join(ch for ch in v_clean if ch.isdigit())
            if filtered:
                numeric_vals.append(filtered)

    add = len([v for v in numeric_vals if v is not None and v != ""])
    # สร้าง list สำหรับ DATAEXCHANGE และ OR
    dataexchange = []
    or_list = []
    for v in numeric_vals:
        try:
            iv = int(v)
            # DATAEXCHANGE เพิ่มค่า add, pad/truncate ให้ยาว n
            dataexchange.append(str(iv + add).zfill(n)[-n:])
            # OR เก็บเป็น padded string (ตามโค้ดเดิม)
            or_list.append(str(iv).zfill(n)[:n])
        except Exception:
            continue

    return {DATAEXCHANGE: dataexchange, OR: or_list}

# --- end patch ---
